# ⚡ Calgary Household Energy — LSTM Time Series Forecasting
> **Dataset:** [Kaggle — Calgary Household Energy Consumption (Synthetic)](https://www.kaggle.com/datasets/yonathanhary/household-energy-consumption-synthetic/data)  
> **Goal:** Forecast the next **N days** of daily kWh consumption per household using a many-to-one LSTM  
> **Author:** Yonathan Hary Hutagalung

### Notebook Structure
1. Setup & Data Loading
2. Time Series EDA
3. Data Preparation (Sequence Building)
4. LSTM Model Architecture
5. Training & Validation
6. Evaluation & Forecasting
7. Multi-Step Forecast Plot
8. Model Export

---
## 1. Setup & Data Loading

In [ ]:
# Install if needed (Colab / Kaggle)
# !pip install tensorflow scikit-learn pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, joblib
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f'TensorFlow version: {tf.__version__}')
print('Setup complete ✅')

In [ ]:
# ── Load dataset ────────────────────────────────────────────────────────────
# Option A: Kaggle environment
# df_raw = pd.read_csv('/kaggle/input/household-energy-consumption-synthetic/calgary_household_energy_synthetic.csv', parse_dates=['Date'])

# Option B: Local file
df_raw = pd.read_csv('calgary_household_energy_synthetic.csv', parse_dates=['Date'])

df_raw = df_raw.sort_values(['Household_ID', 'Date']).reset_index(drop=True)
print(f'Shape: {df_raw.shape}')
print(f'Households: {df_raw.Household_ID.nunique()}')
print(f'Date range: {df_raw.Date.min()} → {df_raw.Date.max()}')
df_raw.head()

---
## 2. Time Series EDA

In [ ]:
# ── 2.1 Aggregate: daily mean kWh across ALL households ─────────────────────
daily_avg = df_raw.groupby('Date')['Daily_kWh'].mean().reset_index()
daily_avg.columns = ['Date', 'Mean_kWh']

plt.figure(figsize=(15, 4))
plt.plot(daily_avg['Date'], daily_avg['Mean_kWh'], lw=0.9, color='steelblue', alpha=0.85)
plt.title('Average Daily kWh Across All Households (2024–2025)', fontsize=13)
plt.xlabel('Date'); plt.ylabel('Mean Daily kWh')
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.2 Seasonal decomposition proxy: monthly boxplot ───────────────────────
df_raw['Month'] = df_raw['Date'].dt.month

plt.figure(figsize=(12, 4))
sns.boxplot(data=df_raw, x='Month', y='Daily_kWh',
            palette='coolwarm', flierprops=dict(marker='o', markersize=2, alpha=0.3))
plt.title('Monthly Distribution of Daily kWh', fontsize=13)
plt.xticks(ticks=range(12),
           labels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.3 Autocorrelation (ACF) on aggregated daily mean ──────────────────────
from pandas.plotting import autocorrelation_plot

plt.figure(figsize=(12, 4))
autocorrelation_plot(daily_avg['Mean_kWh'])
plt.title('Autocorrelation — Daily Mean kWh', fontsize=13)
plt.xlim(0, 60)
plt.tight_layout(); plt.show()

# Key observation: weekly (lag-7) and seasonal patterns expected for Calgary

In [ ]:
# ── 2.4 Pick a single household to model ────────────────────────────────────
# Strategy: forecast for one representative household;
# extend to all households by looping the same pipeline

HOUSEHOLD = 'HH_001'   # change to any HH_001 ... HH_140

df_hh = df_raw[df_raw['Household_ID'] == HOUSEHOLD].copy()
df_hh = df_hh.set_index('Date').sort_index()

plt.figure(figsize=(15, 3))
plt.plot(df_hh.index, df_hh['Daily_kWh'], lw=0.9, color='darkorange', alpha=0.85)
plt.title(f'Daily kWh — {HOUSEHOLD}', fontsize=13)
plt.xlabel('Date'); plt.ylabel('Daily kWh')
plt.tight_layout(); plt.show()
print(f'Total days: {len(df_hh)}')

---
## 3. Data Preparation (Sequence Building)

In [ ]:
# ── Feature selection for LSTM ──────────────────────────────────────────────
# Use multivariate input: past kWh + exogenous features

LSTM_FEATURES = ['Daily_kWh', 'Outside_Temperature_C', 'Has_EV_Car',
                 'Household_Size', 'Living_Area_m2', 'AC_Hours_Used']
TARGET_COL    = 'Daily_kWh'
LOOKBACK      = 30   # use past 30 days to predict next day
FORECAST_DAYS = 30   # rolling forecast horizon for evaluation

series = df_hh[LSTM_FEATURES].copy()
print(f'Series shape: {series.shape}')
series.head(3)

In [ ]:
# ── Scale features ──────────────────────────────────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(series)

# We need a separate scaler just for the target (Daily_kWh) to inverse-transform predictions
target_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler.fit(series[[TARGET_COL]])

print(f'Scaled array shape: {scaled.shape}')

In [ ]:
# ── Build sequences ──────────────────────────────────────────────────────────
def build_sequences(data, lookback, target_idx=0):
    """
    data       : 2D numpy array (timesteps x features)
    lookback   : number of past days used as input window
    target_idx : column index of the target variable
    Returns X (samples, lookback, features) and y (samples,)
    """
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i, :])       # all features, past lookback days
        y.append(data[i, target_idx])            # next day kWh (scaled)
    return np.array(X), np.array(y)

X_all, y_all = build_sequences(scaled, LOOKBACK)
print(f'X shape: {X_all.shape}  |  y shape: {y_all.shape}')

# Train / Validation / Test split  (70% / 15% / 15%)
n = len(X_all)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train, y_train = X_all[:train_end],       y_all[:train_end]
X_val,   y_val   = X_all[train_end:val_end], y_all[train_end:val_end]
X_test,  y_test  = X_all[val_end:],          y_all[val_end:]

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

---
## 4. LSTM Model Architecture

In [ ]:
# ── Build LSTM ───────────────────────────────────────────────────────────────
N_FEATURES = X_train.shape[2]   # number of input features

def build_lstm(lookback, n_features, units_1=128, units_2=64, dropout=0.2, lr=1e-3):
    model = Sequential([
        Input(shape=(lookback, n_features)),
        LSTM(units_1, return_sequences=True),
        Dropout(dropout),
        LSTM(units_2, return_sequences=False),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dense(1)   # single-step forecast
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

model = build_lstm(LOOKBACK, N_FEATURES)
model.summary()

---
## 5. Training & Validation

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    ModelCheckpoint('lstm_best.keras', monitor='val_loss', save_best_only=True, verbose=0)
]

# ── Train ────────────────────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)
print('Training complete ✅')

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['loss'],     label='Train Loss',      color='steelblue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='darkorange')
axes[0].set_title('Loss (MSE) over Epochs', fontsize=12)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(history.history['mae'],     label='Train MAE',      color='steelblue')
axes[1].plot(history.history['val_mae'], label='Validation MAE', color='darkorange')
axes[1].set_title('MAE over Epochs', fontsize=12)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE (scaled)')
axes[1].legend()

plt.suptitle(f'LSTM Training History — {HOUSEHOLD}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 6. Evaluation & Forecasting

In [ ]:
# ── Load best checkpoint & predict on test set ───────────────────────────────
best_model = load_model('lstm_best.keras')

y_pred_scaled = best_model.predict(X_test, verbose=0).flatten()

# Inverse transform (target column only)
y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Metrics
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)

print('=== LSTM Test Set Results ===')
print(f'RMSE : {rmse:.3f}')
print(f'MAE  : {mae:.3f}')
print(f'R²   : {r2:.4f}')

In [ ]:
# ── Actual vs Predicted — test period ────────────────────────────────────────
test_dates = df_hh.index[LOOKBACK + val_end:]

plt.figure(figsize=(14, 4))
plt.plot(test_dates, y_true, label='Actual',    color='steelblue', lw=1.2)
plt.plot(test_dates, y_pred, label='Predicted', color='darkorange', lw=1.2, alpha=0.85)
plt.fill_between(test_dates, y_true, y_pred, alpha=0.15, color='gray')
plt.title(f'LSTM — Actual vs Predicted kWh (Test Set) — {HOUSEHOLD}', fontsize=13)
plt.xlabel('Date'); plt.ylabel('Daily kWh')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── Residuals ────────────────────────────────────────────────────────────────
residuals = y_true - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(y_pred, residuals, alpha=0.4, s=20, color='steelblue')
axes[0].axhline(0, color='red', lw=1.5, ls='--')
axes[0].set_title('Residuals vs Predicted', fontsize=12)
axes[0].set_xlabel('Predicted kWh'); axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].set_title('Residual Distribution', fontsize=12)
axes[1].set_xlabel('Residual (kWh)')

plt.tight_layout(); plt.show()

---
## 7. Multi-Step Forecast Plot

In [ ]:
# ── Recursive multi-step forecast (FORECAST_DAYS ahead) ──────────────────────
# Uses last LOOKBACK days of the full series as the seed window

last_window = scaled[-LOOKBACK:].copy()   # shape: (LOOKBACK, N_FEATURES)
future_preds_scaled = []

for _ in range(FORECAST_DAYS):
    x_input = last_window.reshape(1, LOOKBACK, N_FEATURES)
    pred_s  = best_model.predict(x_input, verbose=0)[0, 0]
    future_preds_scaled.append(pred_s)
    # Shift window: drop oldest, append new prediction (keep other features constant = last known)
    new_row          = last_window[-1].copy()
    new_row[0]       = pred_s           # update only the target column (index 0)
    last_window      = np.vstack([last_window[1:], new_row])

# Inverse transform
future_preds = target_scaler.inverse_transform(
    np.array(future_preds_scaled).reshape(-1, 1)
).flatten()

# Future date index
last_date    = df_hh.index[-1]
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=FORECAST_DAYS, freq='D')

print(f'Forecast horizon: {FORECAST_DAYS} days')
print(f'Forecast period : {future_dates[0].date()} → {future_dates[-1].date()}')

In [ ]:
# ── Plot: historical + forecast ──────────────────────────────────────────────
HISTORY_PLOT_DAYS = 60   # show last 60 days of history for context

hist_dates = df_hh.index[-HISTORY_PLOT_DAYS:]
hist_vals  = df_hh['Daily_kWh'].values[-HISTORY_PLOT_DAYS:]

plt.figure(figsize=(15, 5))
plt.plot(hist_dates,   hist_vals,    label='Historical',       color='steelblue', lw=1.4)
plt.plot(future_dates, future_preds, label=f'{FORECAST_DAYS}-Day Forecast',
         color='darkorange', lw=2, ls='--', marker='o', markersize=4)
plt.axvline(last_date, color='gray', lw=1.2, ls=':', label='Forecast Start')
plt.fill_between(future_dates,
                 future_preds * 0.88, future_preds * 1.12,   # ±12% confidence band
                 alpha=0.15, color='darkorange', label='±12% Confidence Band')
plt.title(f'LSTM {FORECAST_DAYS}-Day Ahead Forecast — {HOUSEHOLD}', fontsize=13, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Daily kWh')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── Forecast table ───────────────────────────────────────────────────────────
forecast_df = pd.DataFrame({
    'Date':          future_dates,
    'Predicted_kWh': future_preds.round(3),
    'Est_Cost_CAD':  (future_preds * 0.1206).round(4)
})
print(forecast_df.to_string(index=False))

---
## 8. Model Export

In [ ]:
# ── Save model & scaler ──────────────────────────────────────────────────────
best_model.save('lstm_model.keras')
joblib.dump(scaler,        'lstm_scaler.pkl')
joblib.dump(target_scaler, 'lstm_target_scaler.pkl')

print('Saved:')
print('  → lstm_model.keras')
print('  → lstm_scaler.pkl')
print('  → lstm_target_scaler.pkl')

# Verify
verify_model  = load_model('lstm_model.keras')
verify_scaler = joblib.load('lstm_scaler.pkl')
print('\nVerification load: OK ✅')

# Final summary
print(f'\n=== Final LSTM Results ({HOUSEHOLD}) ===')
print(f'RMSE : {rmse:.3f}')
print(f'MAE  : {mae:.3f}')
print(f'R²   : {r2:.4f}')
print(f'Lookback window : {LOOKBACK} days')
print(f'Forecast horizon: {FORECAST_DAYS} days')
print(f'Architecture    : LSTM(128) → Dropout → LSTM(64) → Dropout → Dense(32) → Dense(1)')